# Installs if needed

In [14]:
# !pip install glob
# !pip install pandas
# !pip install numpy
# !pip install matplotlib
# !pip install seaborn
# !pip install fastparquet
# !pip install sklearn

In [15]:
# imports
import pandas as pd
import numpy as np
import glob
import pyarrow

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

# Data Engineering Pipeline 
### 3. Extract, Transform, Load (ETL)
---
Extracting data from all the files. Combine everything into one dataframe.

All the dataframes that were created from the CSV, JSON, and Parquet files. They will now be combined and be ready to be cleaned.

In [16]:
# Load all CSV Files
csv_files = glob.glob('final_project_data_sp2026_L/*.csv')
df_csv = pd.concat((pd.read_csv(f) for f in csv_files), ignore_index=True)

# Load all JSON Files
json_files = glob.glob('final_project_data_sp2026_L/*.json')
df_json = pd.concat((pd.read_json(f, lines=True, orient='records') for f in json_files), ignore_index=True)

# Load Parquet Files
parquet_files = glob.glob('final_project_data_sp2026_L/*.parquet')
df_parquet = pd.concat((pd.read_parquet(f, engine='fastparquet') for f in parquet_files), ignore_index=True)

# check all dataframe shapes
print("CSV DataFrame:")
print(df_csv.shape)

print("\nJSON DataFrame:")
print(df_json.shape)

print("\nParquet DataFrame:")
print(df_parquet.shape)    


CSV DataFrame:
(4004, 79)

JSON DataFrame:
(27805, 79)

Parquet DataFrame:
(31320, 79)


In [17]:
"""
Combine all the data into a single DataFrame for analysis and modeling. This will allow us to clean up the data. 
"""
total_df = pd.concat([df_csv, df_json, df_parquet], ignore_index=True)

# check the combined dataframe
print("\nCombined DataFrame:")
print(total_df.shape[0], "rows,", total_df.shape[1], "columns")



Combined DataFrame:
63129 rows, 79 columns


### 4. Data Transformation 
---
...

In [18]:
"""
Remove all the duplicates that are in the combined dataframe. 
"""

print("\nBefore duplicate removal:", total_df.shape)
total_df = total_df.drop_duplicates()
print("After duplicate removal:", total_df.shape)



Before duplicate removal: (63129, 79)
After duplicate removal: (49366, 79)


In [ ]:
"""
Handle all the missing values in the combined dataframe. 
"""
# Check for missing values
print("\nMissing values in each column:", total_df.isnull().sum().sort_values(ascending=False))

# fill in the numerical  columns with median values
num_cols = total_df.select_dtypes(include=[np.number]).columns
total_df[num_cols] = total_df[num_cols].fillna(total_df[num_cols].median())

# fill categorical columns with mode values
cat_cols = total_df.select_dtypes(include=['object']).columns
for col in cat_cols:
    total_df[col] = total_df[col].fillna(total_df[col].mode()[0])


Missing values in each column: Flow Bytes/s            4
 Flow Packets/s         2
 Destination Port       0
 Average Packet Size    0
 Fwd Avg Bulk Rate      0
                       ..
 Bwd IAT Std            0
 Bwd IAT Mean           0
Bwd IAT Total           0
 Fwd IAT Min            0
 Label                  0
Length: 79, dtype: int64


In [21]:
"""
Remove outliers from the combined dataframe using IQR method
"""
def iqr_outlier_removal(df, column):
    for col in column: 
        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)
        IQR = Q3 - Q1
        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR
        df = df[(df[col] >= lower_bound) & (df[col] <= upper_bound)]
    return df

total_df = iqr_outlier_removal(total_df, num_cols)

In [22]:
"""
Feature Scaling
"""
scaler = StandardScaler()
total_df[num_cols] = scaler.fit_transform(total_df[num_cols])   